# Deploying NVIDIA Nemotron-3-Ultra with TensorRT-LLM

This notebook will walk you through how to run the `nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B` model via TensorRT-LLM.

[TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) is NVIDIA's open-source library for accelerating and optimizing LLM inference performance on NVIDIA GPUs.

For more details on the model [click here](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16).

Prerequisites:
- NVIDIA GPUs with recent drivers and CUDA 12.x (Blackwell architecture required for both variants)
    - BF16: 8x GB200/B200/B300 | NVFP4: 4x GB200/B200/B300/GB300
- Python 3.10+
- Docker

## Overview

- **Serve** the Nemotron-3-Ultra model using TensorRT-LLM
- **Query the model** through an OpenAI-compatible REST API
- **Invoke tools** using structured function-calling outputs
- **Tune reasoning depth** by configuring the model's thinking budget

## Table of Contents

1. **Prerequisites & environment** - Launch Docker container
2. **Verify GPU** - Confirm CUDA and GPU availability
3. **OpenAI-compatible server** - Launch and query TensorRT-LLM
   - **Create a YAML file** - Model configuration
   - **Load the model** - BF16 and NVFP4 variants
   - **Use the API** - Chat completions and streaming
   - **Tool calling** - Function calling via OpenAI tools schema
   - **Controlling Reasoning Budget** - Limit reasoning trace length
4. **Cleanup and shutdown**

## Prerequisites & environment

### Launch the Docker container

Open a terminal on the host and start an interactive shell inside the TensorRT-LLM container.

```shell
docker run --rm -it --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --gpus=all \
  -p 8000:8000 \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc17
```

> **Note:** Mount the HuggingFace cache directory so model weights are read from disk rather than re-downloaded on each run. Replace `~/.cache/huggingface` if your cache is in a different location.

> **Note:** Current TRT-LLM support for Nemotron-3-Ultra is limited to Blackwell architecture (B200/B300 and GB200/GB300).

All `trtllm-serve` commands later in this notebook should be run from inside this container.

In [ ]:
#If pip not found
!python3 -m ensurepip --default-pip

In [ ]:
%pip install torch==2.9.1 openai transformers

## Verify GPU

Check that CUDA is available and the GPUs are detected correctly.

> **Expected output:** `CUDA available: True` with 8 B200 GPUs listed. If CUDA is `False`, check your driver installation.

In [1]:
import sys
import torch

print(f"Python: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

Python: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
CUDA available: True
Num GPUs: 8
GPU[0]: NVIDIA B200
GPU[1]: NVIDIA B200
GPU[2]: NVIDIA B200
GPU[3]: NVIDIA B200
GPU[4]: NVIDIA B200
GPU[5]: NVIDIA B200
GPU[6]: NVIDIA B200
GPU[7]: NVIDIA B200


## OpenAI-compatible server

Start a local OpenAI-compatible server with TensorRT-LLM from inside the Docker container terminal.

### Create a YAML file with the required configuration

Run the appropriate config for your variant from the Docker terminal before serving.

#### BF16

```shell
cat > ./extra-llm-api-config.yml << EOF
cuda_graph_config:
  enable_padding: true
  max_batch_size: 256

enable_chunked_prefill: true
max_batch_size: 256
max_num_tokens: 32768
num_postprocess_workers: 4

kv_cache_config:
  enable_block_reuse: false
  free_gpu_memory_fraction: 0.75
  dtype: fp8

moe_config:
  backend: TRTLLM
EOF
```

#### NVFP4

```shell
cat > ./extra-llm-api-config.yml << EOF
cuda_graph_config:
  enable_padding: true
  max_batch_size: 16

enable_chunked_prefill: true
enable_attention_dp: true
max_num_tokens: 16384
num_postprocess_workers: 4

kv_cache_config:
  dtype: fp8
  enable_block_reuse: false
  free_gpu_memory_fraction: 0.7
  mamba_ssm_cache_dtype: float16
  mamba_ssm_philox_rounds: 5
  mamba_ssm_stochastic_rounding: true

moe_config:
  backend: CUTEDSL

speculative_config:
  decoding_type: MTP
  max_draft_len: 3
  num_nextn_predict_layers: 3
  allow_advanced_sampling: true
EOF
```

### Configuration reference

| | BF16 | NVFP4 |
|---|---|---|
| **Model** | `nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16` | `nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-NVFP4` |
| **Supported hardware** | 8x GB200/B200/B300 (Blackwell only) | 4x GB200/B200/B300/GB300 (Blackwell only) |
| **Docker image** | `nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc17` | `nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc17` |
| **TP / EP (this notebook)** | 8 / 1 | 4 / 4 |
| **Reasoning parser** | `nano-v3` | `nano-v3` |
| **Tool parser** | `qwen3_coder` | `qwen3_coder` |
| **Port** | 8000 | 8000 |

### Load the model

Run one of the following commands from inside the Docker terminal.

> **Note:** Parser names are backend-specific and not interchangeable. TensorRT-LLM uses `--reasoning_parser nano-v3` and `--tool_parser qwen3_coder`. vLLM uses `--reasoning-parser nemotron_v3` and SGLang uses `--reasoning-parser nemotron_3` — these are different identifiers for the same logical capability.

#### BF16 (8x B200)

```shell
TLLM_ALLOW_LONG_MAX_MODEL_LEN=1 trtllm-serve nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16 \
  --host 0.0.0.0 \
  --port 8000 \
  --backend pytorch \
  --max_batch_size 256 \
  --tp_size 8 --ep_size 1 \
  --max_num_tokens 32768 \
  --trust_remote_code \
  --served_model_name nemotron-3-ultra \
  --reasoning_parser nano-v3 \
  --tool_parser qwen3_coder \
  --extra_llm_api_options extra-llm-api-config.yml
```

#### NVFP4 (4x B200)

```shell
trtllm-serve nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-NVFP4 \
  --host 0.0.0.0 \
  --port 8000 \
  --backend pytorch \
  --max_batch_size 16 \
  --tp_size 4 --ep_size 4 \
  --max_num_tokens 16384 \
  --trust_remote_code \
  --served_model_name nemotron-3-ultra \
  --reasoning_parser nano-v3 \
  --tool_parser qwen3_coder \
  --extra_llm_api_options extra-llm-api-config.yml
```

### Wait for the server to be ready

Poll `/v1/models` from a host terminal before sending any requests:

```shell
until curl -sf http://localhost:8000/v1/models > /dev/null 2>&1; do
  echo "Waiting for server..."; sleep 10
done
echo "Server is ready"
```

> **Expected output:** The loop prints `Waiting for server...` during model loading, then exits with `Server is ready` once the endpoint responds.

> **Note:** On the first run, the model weights will be downloaded from Hugging Face before loading begins. For a large model like this, the combined download and load time can be significant — this is expected behavior.

### Generate responses

The cells below show single, batched, and streamed completions, followed by reasoning on/off, tool calling, and reasoning budget examples.

In [3]:
from openai import OpenAI

# Set this to match the --served_model_name used when starting the server
SERVED_MODEL_NAME = "nemotron-3-ultra"
BASE_URL = "http://localhost:8000/v1"

client = OpenAI(base_url=BASE_URL, api_key="null")

In [4]:
# Single chat completion
response = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 bullet points about TensorRT-LLM."},
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
)
print(response.choices[0].message.content)

*   **High-Performance Inference Engine:** It is an open-source library from NVIDIA that accelerates Large Language Model (LLM) inference on NVIDIA GPUs using techniques like tensor parallelism, pipeline parallelism, in-flight batching, and custom kernels (FlashAttention, fused MLP).
*   **Advanced Quantization & Optimization:** It supports post-training quantization (PTQ) and quantization-aware training (QAT) for INT8, INT4, and FP8 precision, alongside weight-only quantization, significantly reducing memory footprint and latency with minimal accuracy loss.
*   **Production-Ready Deployment Ecosystem:** It provides a Python API for model definition/optimization, a C++ runtime for low-overhead execution, and native integration with **Triton Inference Server** for dynamic batching, model ensembling, and scalable serving in production environments.


### Batched completions

Send multiple prompts in sequence and collect all responses.

In [4]:
prompts = [
    "What is the square root of 144?",
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
]

for prompt in prompts:
    response = client.chat.completions.create(
        model=SERVED_MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=1.0,
        top_p=0.95,
        max_tokens=256,
    )
    print(f"Q: {prompt}")
    print(f"A: {response.choices[0].message.content}\n")

Q: What is the square root of 144?
A: The square root of 144 is **12**.

Q: What is the capital of France?
A: The capital of France is **Paris**.

Q: Explain quantum computing in simple terms.
A: Imagine you’re in a giant library looking for one specific book.

**A classical computer (like the one you’re using now)** is like a librarian who runs incredibly fast. They check books one by one, shelf by shelf, until they find the right one. They are super speedy, but they can only be in one place at a time.

**A quantum computer** is like a magical librarian who can look at **all the books at the exact same time.**

Here are the three main "magic tricks" that make this possible:

### 1. Bits vs. Qubits (The Coin Analogy)
*   **Classical Bit:** Think of a coin lying flat on a table. It is either **Heads (1)** or **Tails (0)**. It’s definitely one or the other.
*   **Qubit (Quantum Bit):** Think of a coin **spinning on the table**. While it’s spinning, is it Heads or Tails? It’s actually a b

### Streamed generation

Receive tokens as they are generated using the OpenAI streaming API.

In [5]:
# Streaming chat completion
print("Streaming response:")
stream = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    stream=True,
)
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

Streaming response:
The first 5 prime numbers are:

**2, 3, 5, 7, 11**

### Reasoning

The model supports two modes — **Reasoning ON** (default) and **Reasoning OFF**.

Toggle by setting `enable_thinking` to `False` in `chat_template_kwargs`. Use `temperature=1.0, top_p=0.95` when reasoning is on, and `temperature=0.2` when it is off.

In [6]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
)
print("Reasoning:", resp.choices[0].message.reasoning_content)
print("Content:", resp.choices[0].message.content)
print()

# Reasoning off
print("Reasoning off")
resp2 = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 bullet points about TensorRT-LLM."}
    ],
    temperature=0.2,
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
print(resp2.choices[0].message.content)

Reasoning on
Reasoning: The user wants a haiku about GPUs (Graphics Processing Units).
A haiku has a 5-7-5 syllable structure.
I need to capture the essence of GPUs: parallel processing, graphics, AI, heat, speed.
Content: Thousands of cores,
Crunching numbers in parallel,
Pixels bloom to life.

Reasoning off
*   **High-Performance Inference Engine:** It is an open-source library from NVIDIA designed to accelerate and optimize Large Language Model (LLM) inference on NVIDIA GPUs, delivering significant speedups over standard frameworks like PyTorch.
*   **Advanced Optimization Techniques:** It implements key LLM-specific optimizations including quantization (FP8, INT4), kernel fusion, in-flight batching, paged attention, and speculative decoding to maximize throughput and minimize latency.
*   **Seamless Model Support & Deployment:** It provides a Python API to easily convert models from Hugging Face or other sources into optimized TensorRT engines and integrates with Triton Inference S

### Tool calling

Call functions using the OpenAI Tools schema and inspect the returned `tool_calls`.

In [7]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_total": {
                        "type": "integer",
                        "description": "The total amount of the bill"
                    },
                    "tip_percentage": {
                        "type": "integer",
                        "description": "The percentage of tip to be applied"
                    }
                },
                "required": ["bill_total", "tip_percentage"]
            }
        }
    }
]

completion = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}
    ],
    tools=TOOLS,
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
    stream=False
)

print("Reasoning:", completion.choices[0].message.reasoning_content)
print("Tool calls:", completion.choices[0].message.tool_calls)

Reasoning: The user is asking: "My bill is $50. What will be the amount for 15% tip?" They want to compute tip for $50 at 15%. We have a function calculate_tip that takes bill_total (integer) and tip_percentage (integer). It likely returns tip amount. We'll call calculate_tip with bill_total=50, tip_percentage=15.
Tool calls: [ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-1e015b925579496cb7e5664c169cb5cf', function=Function(arguments='{"bill_total": 50, "tip_percentage": 15}', name='calculate_tip'), type='function')]


### Controlling Reasoning Budget

The `reasoning_budget` parameter lets you limit how long the model reasons before producing a response. When the reasoning trace reaches the token budget, the model will try to wrap up at the next newline.

> **Note:** If no newline is encountered within 500 tokens after the budget threshold, the reasoning trace is forcibly terminated at `reasoning_budget + 500` tokens.

In [8]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        response = self.client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=reasoning_budget,
            **kwargs
        )

        reasoning_content = response.choices[0].message.reasoning_content or ""

        if "</think>" not in reasoning_content:
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"

        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used

        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )

        response = self.client.completions.create(
            model=model,
            prompt=prompt,
            max_tokens=remaining_tokens,
            **kwargs
        )

        return {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }

In [6]:
budget_client = ThinkingBudgetClient(
    base_url="http://localhost:8000/v1",
    api_key="null",
    tokenizer_name_or_path="nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16"  # use actual HF model ID for tokenizer
)

In [7]:
resp = budget_client.chat_completion(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    max_tokens=512,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"])
print("Content:", resp["content"])

Reasoning: The user wants a haiku about GPUs.
A haiku has a 5-7-5 syllable structure.
Topic: GPUs (Graphics Processing Units), parallel processing, gaming, AI, rendering..
Content: Thousands of cores,
Parallel dreams rendered fast,
Pixels bloom to life.


## Cleanup and shutdown

To tear down this TensorRT-LLM workflow:

1. In the terminal running `trtllm-serve`, press `Ctrl+C` to stop the server.
2. In the Docker shell, run `exit` to stop the container (`--rm` removes it automatically).
3. Optionally run the next cell to clear notebook-side CUDA cache.

In [ ]:
import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Notebook-side CUDA cache cleanup complete.")